In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from skimage import util, exposure
from skimage.io import imread
from skimage.filters import  threshold_triangle, median, sobel, gaussian
from skimage.measure import label,  regionprops_table, regionprops,  moments_central
from skimage.morphology import  disk, remove_small_objects, binary_dilation, remove_small_holes , erosion, closing, skeletonize, binary_closing, disk, binary_opening
from skimage.transform import rotate
from scipy.ndimage import rotate, binary_dilation
from skimage.transform import probabilistic_hough_line
from skimage.draw import line
from skimage.filters import gaussian, threshold_otsu, threshold_sauvola, frangi, sato, threshold_yen, threshold_triangle, threshold_local, threshold_mean, threshold_li, threshold_minimum, try_all_threshold
from skimage import util
from liffile import LifFile
import napari
import warnings
warnings.filterwarnings("ignore")

from skimage.graph import route_through_array

import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import median
from skimage.filters import gaussian, threshold_otsu, sobel_v
from skimage import exposure
from skimage.morphology import binary_closing, disk, remove_small_objects, skeletonize, binary_dilation, binary_erosion
from skimage.measure import label, regionprops
from skimage.graph import route_through_array
from skimage.filters import sato
import numpy as np
from skimage.draw import line as sk_line


In [ ]:
def vertical_brightness_ratio(img, eps=1e-6):
    """
    Measures brightness discontinuity top vs bottom.
    Returns absolute log ratio: abs(log(top/bottom)).
    """
    y = int(img.shape[0] * 0.5)
    a = np.median(img[:y])
    b = np.median(img[y:])
    return float(abs(np.log((a + eps) / (b + eps))))

In [108]:
def extend_path_to_edges(path_mask, k=5, max_gap=30, edge_tol=0):
    """
    Extend a left-to-right path so it touches x=0 and x=W-1, but ONLY if the
    current path endpoints are within `max_gap` pixels of those edges.

    - If already touches both edges (within edge_tol), do nothing.
    - If too far from either edge (> max_gap), do not extend either side.
    """
    m = path_mask.astype(bool).copy()
    H, W = m.shape
    ys, xs = np.nonzero(m)

    if xs.size < 2:
        return m

    left_gap  = int(xs.min())          # distance to x=0
    right_gap = int((W - 1) - xs.max()) # distance to x=W-1

    # already spans (optionally with tolerance)
    if left_gap <= edge_tol and right_gap <= edge_tol:
        return m

    def fit_line(x, y):
        if x.size < 2:
            return 0.0, float(y[0]) if y.size else H / 2
        return np.polyfit(x.astype(float), y.astype(float), 1)

    # ---- LEFT EXTENSION (only if close enough) ----
    if left_gap > edge_tol and left_gap <= max_gap:
        idx = np.argsort(xs)[:min(k, xs.size)]
        a, b = fit_line(xs[idx], ys[idx])
        x0 = int(xs[idx].min())
        y0 = int(np.clip(round(a * x0 + b), 0, H - 1))
        y_edge = int(np.clip(round(b), 0, H - 1))  # x=0
        rr, cc = sk_line(y_edge, 0, y0, x0)
        m[rr, cc] = True

        # ---- RIGHT EXTENSION (only if close enough AND OK on left too) ----
        if right_gap > edge_tol and right_gap <= max_gap:
            idx = np.argsort(xs)[-min(k, xs.size):]
            a, b = fit_line(xs[idx], ys[idx])
            x1 = int(xs[idx].max())
            y1 = int(np.clip(round(a * x1 + b), 0, H - 1))
            y_edge = int(np.clip(round(a * (W - 1) + b), 0, H - 1))
            rr, cc = sk_line(y1, x1, y_edge, W - 1)
            m[rr, cc] = True

    return m

In [186]:
def pathmask_to_yx(path_mask):
    """Return y(x) from a path mask by median y per x, plus xs."""
    ys, xs = np.nonzero(path_mask)
    if xs.size == 0:
        return None, None
    x_min, x_max = xs.min(), xs.max()
    X = np.arange(x_min, x_max + 1)
    Y = np.full_like(X, np.nan, dtype=float)
    for i, x in enumerate(X):
        yy = ys[xs == x]
        if yy.size:
            Y[i] = np.median(yy)
    # fill gaps
    good = np.isfinite(Y)
    if good.sum() >= 2:
        Y = np.interp(X, X[good], Y[good])
    return Y, X

def yx_to_pathmask(Y, X, shape):
    """Build a 1px path mask from y(x)."""
    H, W = shape
    m = np.zeros((H, W), dtype=bool)
    for y, x in zip(Y, X):
        yi = int(np.clip(round(y), 0, H - 1))
        xi = int(np.clip(x, 0, W - 1))
        m[yi, xi] = True
    return m

# # def flatten_big_drop(path_mask, drop_thresh_px=40, tail_frac=0.35):
# #     """
# #     If y(x) has a large jump (drop) in the later part of the trace,
# #     flatten the remainder to a constant y (median of earlier part).
# #     """
# #     H, W = path_mask.shape
# #     Y, X = pathmask_to_yx(path_mask)
# #     if Y is None:
# #         return path_mask

# #     n = len(Y)
# #     if n < 10:
# #         return path_mask

# #     # Look for a big step in the "tail" region (often where the cliff happens)
# #     start_tail = int((1 - tail_frac) * n)
# #     dY = np.abs(np.diff(Y))

# #     tail_dY = dY[start_tail:] if start_tail < len(dY) else dY
# #     if tail_dY.size == 0:
# #         return path_mask

# #     if tail_dY.max() < drop_thresh_px:
# #         return path_mask  # no big drop

# #     # First index where drop exceeds threshold (global index)
# #     k_tail = int(np.argmax(tail_dY))
# #     k = start_tail + k_tail + 1  # +1 because diff

# #     # Flat y = median of "stable" part before the drop
# #     y_flat = float(np.median(Y[:max(1, k)]))

# #     Y2 = Y.copy()
# #     Y2[k:] = y_flat

# #     return yx_to_pathmask(Y2, X, (H, W))
# # def trim_big_drop_either_side_and_extend(
# #     path_mask,
# #     drop_thresh_px=30,
# #     tail_frac=0.35,
# #     extend_k=25,
# #     extend_max_gap=200,
# #     extend_edge_tol=0,
# # ):
# #     """
# #     If y(x) has a large jump near either end, trim that tail and then extend to edges.

# #     - Detect big jump in left tail (first tail_frac of x-range)
# #     - Detect big jump in right tail (last tail_frac of x-range)
# #     - Trim offending tail(s)
# #     - Extend remaining path to edges with extend_path_to_edges

# #     Returns: updated path_mask
# #     """
# #     H, W = path_mask.shape
# #     Y, X = pathmask_to_yx(path_mask)
# #     if Y is None or len(Y) < 10:
# #         return path_mask

# #     n = len(Y)
# #     dY = np.abs(np.diff(Y))

# #     # Tail windows (in terms of diff indices)
# #     left_end = max(1, int(tail_frac * n))          # number of samples to consider on left
# #     right_start = max(0, int((1 - tail_frac) * n)) # start index for right tail in Y

# #     # --- Detect left tail drop (near start) ---
# #     left_drop_idx = None
# #     if left_end - 1 > 0:
# #         left_dY = dY[:left_end - 1]
# #         if left_dY.size and left_dY.max() >= drop_thresh_px:
# #             left_drop_idx = int(np.argmax(left_dY) + 1)  # +1 to map diff->Y index

# #     # --- Detect right tail drop (near end) ---
# #     right_drop_idx = None
# #     if right_start < n - 1:
# #         right_dY = dY[right_start:]
# #         if right_dY.size and right_dY.max() >= drop_thresh_px:
# #             right_drop_idx = int(right_start + np.argmax(right_dY) + 1)

# #     # If nothing to do, just extend (optional)
# #     if left_drop_idx is None and right_drop_idx is None:
# #         return extend_path_to_edges(path_mask, k=extend_k, max_gap=extend_max_gap, edge_tol=extend_edge_tol)

# #     # Decide keep-range [i0:i1)
# #     i0 = 0
# #     i1 = n

# #     # If there's a left drop, drop everything BEFORE that (keep the more stable interior)
# #     if left_drop_idx is not None:
# #         print("LEFT DROP IDENTIFIED")
# #         i0 = max(i0, left_drop_idx)

# #     # If there's a right drop, drop everything AFTER that (keep the more stable interior)
# #     if right_drop_idx is not None:
# #         print("RIGHT DROP IDENTIFIED")
# #         i1 = min(i1, right_drop_idx)

# #     # Guard against empty/degenerate slice
# #     if i1 - i0 < 2:
# #         return extend_path_to_edges(path_mask, k=extend_k, max_gap=extend_max_gap, edge_tol=extend_edge_tol)

# #     Y2 = Y[i0:i1]
# #     X2 = X[i0:i1]
# #     pm2 = yx_to_pathmask(Y2, X2, (H, W))

# #     # Finally extend to edges (your existing method)
# #     pm2 = extend_path_to_edges(pm2, k=extend_k, max_gap=extend_max_gap, edge_tol=extend_edge_tol)
# #     return pm2
# import numpy as np
# from skimage.draw import line as sk_line
# import numpy as np
# from skimage.draw import line as sk_line

# def extend_stable_horizontally_to_edges(
#     pm,
#     Y2, X2,
#     extend_left=True, extend_right=True,
#     x_start_left=None, x_start_right=None,
#     pad_px=5
# ):
#     """
#     Extend the stable region horizontally (constant y) to the image edges.

#     - Uses mean height of the stable region: y_const = mean(Y2)
#     - Starts from (drop_x - pad_px) on the left, and (drop_x + pad_px) on the right, if provided.
#       Otherwise falls back to min/max X2.
#     """
#     H, W = pm.shape
#     out = pm.copy()

#     if Y2 is None or len(Y2) < 2:
#         return out

#     y_const = int(np.clip(np.round(np.mean(Y2)), 0, H - 1))

#     x_min = int(np.clip(np.min(X2), 0, W - 1))
#     x_max = int(np.clip(np.max(X2), 0, W - 1))

#     # Determine start x for extensions
#     if x_start_left is None:
#         xL = x_min
#     else:
#         xL = int(np.clip(x_start_left - pad_px, 0, W - 1))

#     if x_start_right is None:
#         xR = x_max
#     else:
#         xR = int(np.clip(x_start_right + pad_px, 0, W - 1))

#     # Extend left
#     if extend_left and xL > 0:
#         rr, cc = sk_line(y_const, 0, y_const, xL)
#         out[rr, cc] = 1

#     # Extend right
#     if extend_right and xR < (W - 1):
#         rr, cc = sk_line(y_const, xR, y_const, W - 1)
#         out[rr, cc] = 1

#     return out


# def trim_and_extend_with_horizontal(
#     pm,
#     tail_frac=0.15,
#     drop_thresh_px=8,
#     pad_px=5,
#     extend_k=3,
#     extend_max_gap=5,
#     extend_edge_tol=2
# ):
#     H, W = pm.shape
#     Y, X = pathmask_to_yx(pm)
#     if Y is None or len(Y) < 10:
#         return pm

#     n = len(Y)
#     dY = np.abs(np.diff(Y))

#     left_end = max(1, int(tail_frac * n))
#     right_start = max(0, int((1 - tail_frac) * n))

#     # --- Detect left tail drop (near start) ---
#     left_drop_idx = None
#     if left_end - 1 > 0:
#         left_dY = dY[:left_end - 1]
#         if left_dY.size and left_dY.max() >= drop_thresh_px:
#             left_drop_idx = int(np.argmax(left_dY) + 1)  # diff->Y index

#     # --- Detect right tail drop (near end) ---
#     right_drop_idx = None
#     if right_start < n - 1:
#         right_dY = dY[right_start:]
#         if right_dY.size and right_dY.max() >= drop_thresh_px:
#             right_drop_idx = int(right_start + np.argmax(right_dY) + 1)

#     # No drops -> keep your original extend behavior
#     if left_drop_idx is None and right_drop_idx is None:
#         return extend_path_to_edges(pm, k=extend_k, max_gap=extend_max_gap, edge_tol=extend_edge_tol)

#     # Decide keep-range [i0:i1)
#     i0, i1 = 0, n
#     if left_drop_idx is not None:
#         print("LEFT DROP IDENTIFIED")
#         i0 = max(i0, left_drop_idx)
#     if right_drop_idx is not None:
#         print("RIGHT DROP IDENTIFIED")
#         i1 = min(i1, right_drop_idx)

#     # Guard against empty/degenerate slice
#     if i1 - i0 < 2:
#         return extend_path_to_edges(pm, k=extend_k, max_gap=extend_max_gap, edge_tol=extend_edge_tol)

#     # Stable interior
#     Y2 = Y[i0:i1]
#     X2 = X[i0:i1]
#     pm2 = yx_to_pathmask(Y2, X2, (H, W))

#     # Drop x-locations in original arrays (used to start extension)
#     x_drop_left = int(X[left_drop_idx]) if left_drop_idx is not None else None
#     x_drop_right = int(X[right_drop_idx]) if right_drop_idx is not None else None

#     pm2 = extend_stable_horizontally_to_edges(
#         pm2,
#         Y2, X2,
#         extend_left=(left_drop_idx is not None),
#         extend_right=(right_drop_idx is not None),
#         x_start_left=x_drop_left,
#         x_start_right=x_drop_right,
#         pad_px=pad_px
#     )

#     return pm2

import numpy as np
from skimage.draw import line as sk_line

def extend_stable_horizontally_to_edges(
    pm,
    Y2, X2,
    extend_left=True, extend_right=True,
    x_start_left=None, x_start_right=None,
    pad_px=5
):
    """
    Extend the stable region horizontally (constant y) to the image edges.

    - y is the mean height of the stable region: round(mean(Y2))
    - if x_start_left/right are provided (drop locations), extension starts from:
        left:  (x_start_left  - pad_px)
        right: (x_start_right + pad_px)
      else falls back to min/max X2.
    """
    H, W = pm.shape
    out = pm.copy()

    if Y2 is None or len(Y2) < 2:
        return out

    y_const = int(np.clip(np.round(np.mean(Y2)), 0, H - 1))

    x_min = int(np.clip(np.min(X2), 0, W - 1))
    x_max = int(np.clip(np.max(X2), 0, W - 1))

    xL = x_min if x_start_left  is None else int(np.clip(x_start_left  - pad_px, 0, W - 1))
    xR = x_max if x_start_right is None else int(np.clip(x_start_right + pad_px, 0, W - 1))

    # Extend left to x=0
    if extend_left and xL > 0:
        rr, cc = sk_line(y_const, 0, y_const, xL)
        out[rr, cc] = 1

    # Extend right to x=W-1
    if extend_right and xR < (W - 1):
        rr, cc = sk_line(y_const, xR, y_const, W - 1)
        out[rr, cc] = 1

    return out


def trim_and_extend_with_horizontal(pm, tail_frac=0.15, drop_thresh_px=8, pad_px=5,
                                   extend_k=3, extend_max_gap=5, extend_edge_tol=2):
    """
    If a drop is detected in either tail (based on |diff(Y)|), cut off that tail and
    extend the stable region horizontally using the mean Y of the stable segment.

    Note: no fit_k parameter here (avoids your current TypeError).
    """
    H, W = pm.shape
    Y, X = pathmask_to_yx(pm)
    if Y is None or len(Y) < 10:
        return pm

    n = len(Y)
    dY = np.abs(np.diff(Y))

    left_end = max(1, int(tail_frac * n))
    right_start = max(0, int((1 - tail_frac) * n))

    # --- Detect left tail drop ---
    left_drop_idx = None
    if left_end - 1 > 0:
        left_dY = dY[:left_end - 1]
        if left_dY.size and left_dY.max() >= drop_thresh_px:
            left_drop_idx = int(np.argmax(left_dY) + 1)

    # --- Detect right tail drop ---
    right_drop_idx = None
    if right_start < n - 1:
        right_dY = dY[right_start:]
        if right_dY.size and right_dY.max() >= drop_thresh_px:
            right_drop_idx = int(right_start + np.argmax(right_dY) + 1)

    # No drops -> keep your original extend behavior
    if left_drop_idx is None and right_drop_idx is None:
        return extend_path_to_edges(pm, k=extend_k, max_gap=extend_max_gap, edge_tol=extend_edge_tol)

    # Decide keep-range [i0:i1)
    i0, i1 = 0, n
    if left_drop_idx is not None:
        print("LEFT DROP IDENTIFIED")
        i0 = max(i0, left_drop_idx)
    if right_drop_idx is not None:
        print("RIGHT DROP IDENTIFIED")
        i1 = min(i1, right_drop_idx)

    # Guard against degenerate slice
    if i1 - i0 < 2:
        return extend_path_to_edges(pm, k=extend_k, max_gap=extend_max_gap, edge_tol=extend_edge_tol)

    # Stable interior
    Y2 = Y[i0:i1]
    X2 = X[i0:i1]
    pm2 = yx_to_pathmask(Y2, X2, (H, W))

    # Drop x-locations in original arrays
    x_drop_left  = int(X[left_drop_idx])  if left_drop_idx  is not None else None
    x_drop_right = int(X[right_drop_idx]) if right_drop_idx is not None else None

    pm2 = extend_stable_horizontally_to_edges(
        pm2,
        Y2, X2,
        extend_left=(left_drop_idx is not None),
        extend_right=(right_drop_idx is not None),
        x_start_left=x_drop_left,
        x_start_right=x_drop_right,
        pad_px=pad_px
    )

    return pm2


In [187]:


def extract_boundary(img, thr_func, min_size=100, area_threshold=500, opening_size=3): 
    bw_orig = img > thr_func(img)
    bw = remove_small_objects(bw_orig, min_size=min_size)
    bw = remove_small_holes(bw, area_threshold=area_threshold)
    bw = binary_opening(bw, disk(opening_size))


    labels = label(bw)
    if labels.max() > 0:
        props = regionprops(labels)
        biggest = max(props, key=lambda p: p.area)
        bw = (labels == biggest.label)

    # interface line
    path_mask = bw ^ binary_erosion(bw, disk(1))
    lab = label(path_mask)
    if lab.max() > 0:
        props = regionprops(lab)
        longest = max(props, key=lambda p: p.area) # area == number of pixels
        path_mask = (lab == longest.label)
    return path_mask, bw


def find_best_left_right_route(mask, edge_margin=30, step_cost=100.0):
    H, W = mask.shape

    # Candidate endpoints in left/right margin bands
    left_band  = mask[:, :edge_margin]
    right_band = mask[:, W-edge_margin:]

    left_pts  = list(map(tuple, np.argwhere(left_band)))
    right_pts = list(map(tuple, np.argwhere(right_band)))
    right_pts = [(y, W-edge_margin + x) for (y, x) in right_pts]

    # If we didn't get endpoints (mask slightly short), do a tiny dilation and retry
    if (len(left_pts) == 0) or (len(right_pts) == 0):
        m2 = binary_dilation(mask, disk(2))
        left_band  = m2[:, :edge_margin]
        right_band = m2[:, W-edge_margin:]
        left_pts  = list(map(tuple, np.argwhere(left_band)))
        right_pts = list(map(tuple, np.argwhere(right_band)))
        right_pts = [(y, W-edge_margin + x) for (y, x) in right_pts]
    else:
        m2 = mask

    # Cost: prefer mask, penalize background (allows tiny gaps via m2 dilation)
    off_penalty = step_cost * 0.2 # try 0.05–0.5
    cost = np.where(m2, step_cost, step_cost + off_penalty).astype(np.float32)

    # Cheap reduction of endpoints to avoid O(N^2)
    max_candidates = 50
    if len(left_pts) > max_candidates:
        left_pts = [left_pts[i] for i in np.linspace(0, len(left_pts)-1, max_candidates).astype(int)]
    if len(right_pts) > max_candidates:
        right_pts = [right_pts[i] for i in np.linspace(0, len(right_pts)-1, max_candidates).astype(int)]

    best_score = np.inf
    best_path = None

    for s in left_pts:
        for t in right_pts:
            path, _c = route_through_array(cost, s, t, fully_connected=True)
            path_len = len(path)

            off_count = sum(not m2[y, x] for (y, x) in path)
            score = path_len + 200 * off_count  # heavier penalty to stay on m2

            if score < best_score:
                best_score = score
                best_path = path

    path_mask = np.zeros_like(mask, dtype=bool)
    if best_path is not None:
        for y, x in best_path:
            path_mask[y, x] = True
        return path_mask
    else:
        return None






In [ ]:

def find_ridge_through_plane(img_slice, to_plot=True, ratio_cutoff=0.75):
    # # Gate: decide edge-mode vs ridge-mode
    vr = vertical_brightness_ratio(img_slice, split=0.5, robust=True)
    mode = "edge" if vr > ratio_cutoff else "ridge"    
    if mode == "edge": ###########THIS IS THE PART WITH ISSUES FOR LOW SIGNAL
        H, W = img_slice.shape
        # --- First attempt: Li ---
        img_med = median(img_slice, disk(7))
        path_mask, bw = extract_boundary(img_med, threshold_li)
        title = "li"

        # --- Sanity check: boundary too long? ---
        if path_mask.sum() > 1.5 * W:
            title = "minimum"
            path_mask, bw = extract_boundary(img_med, threshold_minimum, min_size = 0, area_threshold = 0, opening_size=0)
        elif path_mask[0, :].any() or path_mask[-1, :].any():
            title = "triangle"
            path_mask, bw = extract_boundary(exposure.adjust_gamma(img_med, gamma=0.2), threshold_triangle, min_size = 0, area_threshold = 0, opening_size=10)
            
            
            
        # path_mask = flatten_big_drop(path_mask, drop_thresh_px=30, tail_frac=0.4)
        
        # path_mask = trim_and_extend_with_horizontal(path_mask, tail_frac = 0.3, drop_thresh_px=30, fit_k=25)

        path_mask = extend_path_to_edges(path_mask,  k=5, max_gap=30, edge_tol=0)


                
    else: ####### ridge mode 
        title="ridge"
        img_med = median(img_slice, disk(7))
        resp = sato(img_med, sigmas=np.linspace(2, 8, 10), black_ridges=True)
        resp = exposure.rescale_intensity(resp)
        thr = threshold_yen(resp)
        bw = resp > thr
        bw = binary_closing(bw, footprint=disk(9))
        bw = remove_small_objects(bw, min_size=600)
        skel = skeletonize(bw)
        lab = label(skel)
        if lab.max() > 0:
            props = regionprops(lab)
            props_sorted = sorted(props, key=lambda p: p.area, reverse=True)
            labels_to_keep = [p.label for p in props_sorted[:3]]
            skel_main = np.isin(lab, labels_to_keep)
        else:
            skel_main = skel

        path_mask = find_best_left_right_route(skel_main)
        if path_mask is None:
            path_mask = find_best_left_right_route(skel)
        # if path_mask is not None:
        #     path_mask = extend_path_to_edges(path_mask,  k=5, max_gap=30, edge_tol=0)
        # path_mask = trim_and_extend_with_horizontal(path_mask, tail_frac = 0.3, drop_thresh_px=30, fit_k=25)
        path_mask = flatten_big_drop(path_mask, drop_thresh_px=50, tail_frac=0.4)

    if to_plot:
        fig, ax = plt.subplots(ncols=3, figsize=(6, 3))
        ax[0].imshow(img_slice, cmap="gray")
        ax[1].imshow(bw, cmap="gray")
        ax[2].imshow(img_slice, cmap="gray")
        ax[0].set_title("input")
        ax[1].set_title(title)
        ax[2].set_title("path")
        if path_mask is not None:
            y, x = np.nonzero(path_mask)
            ax[2].scatter(x, y, s=2, c="red")
        for a in ax:
            a.axis("off")
        plt.suptitle(f"Mode:{mode}, Brightness ratio:{vr}")
        plt.tight_layout()
        plt.show()

    # return path_mask, mode, vr, img_med

In [189]:
folder = Path(r"Z:\Maria Cuende\0_Projects\0_Placenta\Barrier integrity\20260115-22_Exp6")
lif_files = list(folder.glob("*.lif"))
print(len(lif_files))

lif_path = lif_files[1]
image_index = 7

for image_index in [3]:
    dextran_channel = 2
    with LifFile(lif_path) as lif:
        img = lif.images[image_index]
        print(f"Opening image {lif.name}, which has dimensions {getattr(img, 'dims', None)} and shape {img.shape}")
        img = img.asarray()
    dextran_t0 = img[0, dextran_channel, :, :, :]
    print("dextran_t0 shape:", dextran_t0.shape)


    for z in [5,7,12,15,18,20,25]:
        find_ridge_through_plane(dextran_t0[z, :, :], to_plot=True)

2
Opening image Exp6_20260122_Thalidomide-Ibuprofen.lif, which has dimensions ('T', 'C', 'Z', 'Y', 'X') and shape (4, 3, 41, 512, 512)
dextran_t0 shape: (41, 512, 512)


TypeError: trim_and_extend_with_horizontal() got an unexpected keyword argument 'fit_k'